# Tarea: Geometría de Datos en Producción II
## Transformaciones No Lineales y Reducción de Dimensionalidad (Play Store)

**Objetivo:**
El ecosistema de aplicaciones móviles es el ejemplo perfecto de distribuciones asimétricas (el 1% de las apps se lleva el 99% de las descargas) y de exceso de dimensiones (versiones, nombres únicos, fechas). En esta tarea, aplicarán transformaciones logarítmicas para suavizar la asimetría extrema y aplicarán técnicas de reducción de dimensionalidad para purgar variables de alta cardinalidad y redundantes.

**Instrucciones:**
Resuelvan los 4 retos programando en las celdas de código. Respondan a las preguntas analíticas utilizando comentarios o celdas de texto (Markdown) debajo de cada reto.

In [1]:
import pandas as pd
import numpy as np

# Instrucción: Carga tu archivo limpio. 
# Si no lo tienes a la mano, usa el siguiente bloque para descargar y pre-limpiar lo básico:

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

# Mockup de limpieza rápida
df.drop_duplicates(subset=['App'], inplace=True)
df = df[df['Installs'] != 'Free']
df['Installs'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['Reviews'] = df['Reviews'].astype(float)

print("Datos listos. Total de apps preparadas:", len(df))

Datos listos. Total de apps preparadas: 9659


---
### Reto 1: Diagnóstico de Asimetría (El Sesgo del Mercado)
La columna `Reviews` (Cantidad de reseñas) sufre de hiper-asimetría. Hay aplicaciones con 0 reseñas y gigantes como Facebook con casi 80 millones.

1. Calcula la asimetría (`.skew()`) de la columna original `Reviews`.
2. **Pregunta de Análisis:** Escribe qué valor obtuviste. Si un valor ideal para una distribución normal de Machine Learning está entre -1 y 1, ¿qué nos dice este número sobre la forma geométrica de los datos de la Play Store?

In [2]:
# 1. Calcula y muestra el skew de 'Reviews'
asimetria_original = df['Reviews'].skew()

print(asimetria_original)



26.573458755522793


### RESPUESTA A LA PREGUNTA (Escribe tu análisis aquí):
_WOW, esta asimetria es brutal, 26 es un numero muy grande eso quiere decir que el sesgo esta brutalmente a la derecha_


---
### Reto 2: La Cura Logarítmica (`np.log1p`)
Vamos a curar los gradientes del modelo comprimiendo esta variable exponencial.

1. Crea una nueva columna llamada `Reviews_Log` aplicando la transformación `np.log1p()` a la columna original `Reviews`.
2. Vuelve a calcular el `.skew()` sobre esta nueva columna.
3. **Pregunta de Análisis:** ¿Cuál es el nuevo valor de asimetría? Explica técnicamente por qué usamos `log1p` (logaritmo de $x + 1$) en lugar de un `log` tradicional en este dataset. *(Pista: Piensa en las apps que nadie ha descargado ni calificado).*

In [3]:
# 1. Aplica la transformación logarítmica
df['Reviews_Log'] = np.log1p(df['Reviews'])

# 2. Muestra la nueva asimetría
asimetria_log = df['Reviews_Log'].skew()
print(asimetria_log)


0.04623548762321226



### RESPUESTA A LA PREGUNTA:

_Definitivamente cambiar la escala original a logarítmica, hace que un algoritmo realmente pueda leer los datos sin que haya sesgo, incluso si hay outliers_


---
### Reto 3: Auditoría y Purga de Alta Cardinalidad
Si le pasas a una Red Neuronal una columna donde cada fila tiene un texto distinto, la red memorizará el texto en lugar de aprender el patrón (Maldición de la Dimensionalidad).

1. Utiliza el método `.nunique()` en todo el *DataFrame* para auditar la cardinalidad.
2. Observa las columnas `App` (Nombre de la aplicación) y `Current Ver` (Versión actual como "1.0.1", "2.4", etc.).
3. Utiliza `.drop(columns=[...])` para eliminar **definitivamente** estas dos columnas de tu `df`, ya que son identificadores casi únicos que no aportan valor predictivo tabular.

In [4]:
# 1. Audita la cardinalidad
print(df.nunique())

# 2 y 3. Elimina las columnas 'App' y 'Current Ver'
df = df.drop(columns=['App','Current Ver'])

# Comprobación (Verifica que ya no existan)
print("\nColumnas restantes:", df.columns.tolist())

App               9659
Category            33
Rating              39
Reviews           5330
Size               461
Installs            20
Type                 2
Price               92
Content Rating       6
Genres             118
Last Updated      1377
Current Ver       2817
Android Ver         33
Reviews_Log       5330
dtype: int64

Columnas restantes: ['Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Android Ver', 'Reviews_Log']


---
### Reto 4: Reducción por Multicolinealidad (Correlación)
La reducción de dimensionalidad no solo elimina ruido (como los nombres), también elimina variables **redundantes**. Si dos columnas dicen matemáticamente casi lo mismo, tener ambas solo gasta RAM y causa "multicolinealidad".

1. Selecciona solo las columnas numéricas con `df.select_dtypes(include=[np.number])`.
2. Calcula la matriz de correlación con `.corr()`.
3. Observa la correlación entre `Installs` y `Reviews_Log` (o `Reviews`). 
4. **Pregunta de Análisis:** ¿Cuál es el nivel de correlación entre estas dos variables? Como arquitecto de datos, si tuvieras que reducir el dataset al mínimo tamaño posible para un modelo de regresión, ¿eliminarías una de las dos? ¿Por qué?

In [5]:
# 1 y 2. Calcula la correlación de las variables numéricas

# Aplico la misma escala logarítmica a Installs para comparar en igualdad de condiciones
df['Installs_Log'] = np.log1p(df['Installs'])

numeric_values = df.select_dtypes(include=[np.number])
correlacion = numeric_values.corr()
print(correlacion)

# 3. Granularidad: Installs viene en rangos (cubetas), Reviews es un conteo real
print("\nValores únicos Installs:", df['Installs'].nunique())
print("Valores únicos Reviews :", df['Reviews'].nunique())

# Dentro de una sola cubeta de Installs, cuánto varía Reviews
cubeta = df[df['Installs'] == 1_000_000]['Reviews']
print(f"\nApps con Installs = 1,000,000+: n={len(cubeta)}, "
      f"Reviews min={cubeta.min():.0f}, max={cubeta.max():.0f}, mediana={cubeta.median():.0f}")


                Rating   Reviews  Installs  Reviews_Log  Installs_Log
Rating        1.000000  0.055081  0.040143     0.182837      0.085011
Reviews       0.055081  1.000000  0.625165     0.239724      0.207743
Installs      0.040143  0.625165  1.000000     0.263811      0.263708
Reviews_Log   0.182837  0.239724  0.263811     1.000000      0.958784
Installs_Log  0.085011  0.207743  0.263708     0.958784      1.000000

Valores únicos Installs: 20
Valores únicos Reviews : 5330

Apps con Installs = 1,000,000+: n=1417, Reviews min=1, max=376223, mediana=21892


### RESPUESTA A LA PREGUNTA:
_En la matriz, Reviews e Installs en su escala original ya tienen una correlación de 0.625, que es alta, pero si le aplicamos la misma escala logarítmica a Installs (Installs_Log) la correlación con Reviews_Log sube a 0.959, o sea casi perfecta. Esto pasa porque en la escala original los outliers como Facebook o WhatsApp aplastan la relación, y al comprimir las dos columnas con el logaritmo se ve la relación real._

_Las dos columnas presentan la misma información: qué tanta popularidad tiene la aplicación. Tiene lógica, porque las reviews las dejan usuarios que ya instalaron la app, entonces a más instalaciones, más reviews. Tener las dos en un modelo de regresión es redundante y causa multicolinealidad: el modelo no sabe a cuál de las dos darle el peso y los coeficientes salen inestables. Así que sí, eliminaría una de las dos._

_Ahora, cuál eliminar está un poco complicado, porque honestamente ambas pueden ser buenas opciones:_

- _**Quedarme con Installs:** es la variable de mayor peso, es la medida directa de las descargas, mientras que Reviews depende de cuánta gente decide calificar. Es lo que un negocio entiende directamente._
- _**Quedarme con Reviews_Log:** Installs solo tiene 20 valores distintos, porque la Play Store reporta descargas en rangos (1,000+, 10,000+, etc.), prácticamente es una variable categórica. Reviews es un conteo real con más de 5,000 valores distintos. Dentro de una misma cubeta de Installs (por ejemplo 1,000,000+) hay apps con 1 review y apps con 376,000; Installs no las distingue, Reviews sí. Para una regresión esa resolución es información que se perdería._

_Si tuviera que decidir, me inclinaría por Reviews_Log por la granularidad, pero depende de qué quiera predecir el modelo: si el target fuera Installs, obviamente no se puede eliminar y Reviews pasaría a ser un feature._

_Aparte, ninguna de las dos dice nada de calidad: Reviews_Log correlaciona apenas 0.18 con Rating e Installs 0.04. Una aplicación puede tener muchas o pocas reviews y eso no define si es buena o no; para eso está Rating._
